# GraphRAG: AI Copyright & Governance Knowledge Graph

This notebook builds a full **GraphRAG pipeline** on a pre-scraped dataset of news articles
about AI copyright, governance, and intellectual property.

### Pipeline overview

| Step | What happens |
|---|---|
| 1 | Load and prepare your scraped article dataset |
| 2 | Define a domain ontology for AI copyright/governance |
| 3 | Extract entities + relationships WITH descriptions using a custom extractor |
| 4 | Build a knowledge graph with community detection baked in |
| 5 | Generate LLM community summaries |
| 6 | Visualize the graph interactively |
| 7 | Query using GraphRAG |

> **Refactored implementation:** The explanatory flow below is preserved from
> the original notebook. The implementation now lives in four reusable modules:
> `graph_rag_schema.py`, `graph_rag_engine.py`, `graph_rag_services.py`, and
> `graph_rag_manager.py`. Notebook cells call those classes instead of repeating
> their implementation.

---
## 0. Install Dependencies

In [1]:
# %pip install -r requirements.txt

---
## 1. Imports & Configuration


In [2]:
from pathlib import Path

import nest_asyncio
import pandas as pd

from src import (
    ExtractedEntity,
    ExtractedRelationship,
    ExtractionResult,
    GraphRAGExtractor,
    GraphRAGManager,
    GraphRAGQueryEngine,
    GraphRAGSchema,
    GraphRAGService,
    GraphRAGStore,
)

nest_asyncio.apply()
print("✅ Imports ready")

✅ Imports ready


---
## 2. Configuration

Set `USE_QWEN` in the next cell to switch the complete pipeline between the local Qwen model and OpenAI. Each provider uses separate output files so their graphs can be compared safely.

| Provider | Extraction and community summaries | Final query synthesis |
|---|---|---|
| Qwen | `Qwen3.8-27B` through the local LiteLLM endpoint | `Qwen3.8-27B` |
| OpenAI | `gpt-5-nano` | `gpt-5.6-luna` |


In [ ]:
PROJECT_ROOT = Path.cwd()
USE_QWEN = True  # Set to False to use OpenAI
LLM_PROVIDER = "qwen" if USE_QWEN else "openai"
EXTRACTION_MODEL = "Qwen3.8-27B" if USE_QWEN else "gpt-5-nano"
QUERY_MODEL = "Qwen3.8-27B" if USE_QWEN else "gpt-5.6-luna"
QWEN_BASE_URL = "http://192.168.10.45:4000/v1"

DATASET_FILE = PROJECT_ROOT / "ai_copyright_dataset.csv"
CHECKPOINT_FILE = PROJECT_ROOT / f"ai_copyright_graph_store_{LLM_PROVIDER}.pkl"
GRAPH_DATA_FILE = PROJECT_ROOT / f"graph_data_{LLM_PROVIDER}.json"
GRAPH_TEMPLATE_FILE = PROJECT_ROOT / "graph_template.html"
GRAPH_OUTPUT_FILE = PROJECT_ROOT / f"ai_copyright_graph_{LLM_PROVIDER}.html"

MAX_ARTICLES        = 3    # Number of articles to process
MAX_PATHS_PER_CHUNK = 20    # Max entity-relationship-entity triplet|s extracted per chunk
NUM_WORKERS         = 2     # Lower concurrency reduces API timeouts during structured extraction
MAX_CLUSTER_SIZE    = 10    # Max entities per community cluster (controls how big the clusters should be)
REQUEST_TIMEOUT     = 180.0 # Seconds allowed for each LLM request
REQUEST_MAX_RETRIES = 5     # Retries transient API/network failures with backoff

manager = GraphRAGManager(
    provider=LLM_PROVIDER,
    extraction_model=EXTRACTION_MODEL,
    query_model=QUERY_MODEL,
    qwen_base_url=QWEN_BASE_URL,
    max_paths_per_chunk=MAX_PATHS_PER_CHUNK,
    num_workers=NUM_WORKERS,
    max_cluster_size=MAX_CLUSTER_SIZE,
    request_timeout=REQUEST_TIMEOUT,
    request_max_retries=REQUEST_MAX_RETRIES,
)

EXTRACTION_LLM = manager.extraction_llm
QUERY_LLM = manager.query_llm

print(
    f"✅ Using {LLM_PROVIDER}: {EXTRACTION_LLM.model} for extraction, "
    f"{QUERY_LLM.model} for querying"
)

---
## 3. Define the Ontology

The ontology is the **schema of our knowledge graph**. It controls exactly what
kinds of entities and relationships the LLM is allowed to extract.

### Why it matters

Without a defined schema, the LLM will:
- Invent different entity types per document ("AI_COMPANY" vs "TECH_FIRM" vs "ORGANIZATION")
- Use inconsistent relationship labels for the same concept
- Create duplicate nodes for the same entity under slightly different names

The ontology constrains extraction to a fixed vocabulary, so the graph stays
consistent across all documents.

### For this particular use case - AI copyright & governance

I defined below 7 entity types, 8 relationship type.

But this really depends on your use case and what you want to research.


In [4]:
ENTITY_TYPES = GraphRAGSchema.ENTITY_TYPES
RELATION_TYPES = GraphRAGSchema.RELATION_TYPES

print("✅ Ontology:")
print(f"   Entity types:       {ENTITY_TYPES}")
print(f"   Relationship types: {RELATION_TYPES}")

✅ Ontology:
   Entity types:       ('ORGANIZATION', 'PERSON', 'LEGISLATION', 'LEGAL_CASE', 'CONCEPT', 'GOVERNMENT', 'AI_SYSTEM')
   Relationship types: ('FILED_AGAINST', 'DEFENDANT_IN', 'REGULATES', 'ADVOCATES_FOR', 'TRAINED_ON', 'PART_OF', 'REFERENCES', 'OPPOSES')


---
## 4. Extraction Prompt

This is where our ontology gets embedded into the LLM's instructions.

Most GraphRAG implementations extract bare triplets: `(OpenAI, DEFENDANT_IN, NYT v. OpenAI)`.

This prompt extracts **descriptions alongside every entity and relationship**:
- Entity description: *"OpenAI is an AI research company that developed GPT-4, currently facing multiple copyright lawsuits"*
- Relationship description: *"OpenAI is named as defendant in the New York Times lawsuit over allegedly using copyrighted articles as training data"*

These descriptions flow through to the community summaries, making them far richer
and enabling much better final answers.


In [5]:
KG_TRIPLET_EXTRACT_TMPL = GraphRAGSchema.extraction_prompt()

print("✅ Extraction prompt ready")
print(f"\nPreview (first 300 chars):\n{KG_TRIPLET_EXTRACT_TMPL[:300]}...")

✅ Extraction prompt ready

Preview (first 300 chars):

-Goal-
Given a news article about AI copyright, governance, or intellectual property,
identify all entities mentioned in the article and their relationships.

Extract up to {max_knowledge_triplets} entity-relation triplets.

-Allowed Entity Types-
ORGANIZATION, PERSON, LEGISLATION, LEGAL_CASE, CONC...


---
## 5. Pydantic Extraction Models

Instead of parsing raw LLM text with regex, we define Pydantic models that describe
exactly what we want the LLM to return. LlamaIndex passes these to OpenAI as a
function schema, so the response is structured JSON — validated and typed automatically.

Three models are defined:

- **`ExtractedEntity`** — a single entity with `name`, `type`, and `description`
- **`ExtractedRelationship`** — a relationship between two entities: `source`, `target`, `relation`, `description`
- **`ExtractionResult`** — the top-level wrapper containing a list of each

`EntityTypeStr` and `RelationTypeStr` are `Literal` types built from the ontology defined in the previous cell.
This means the LLM *cannot* return an invalid type — Pydantic will reject it before it ever reaches our code.

In [6]:
print("✅ Pydantic extraction models:")
print("  ", ExtractedEntity.__name__)
print("  ", ExtractedRelationship.__name__)
print("  ", ExtractionResult.__name__)

✅ Pydantic extraction models:
   ExtractedEntity
   ExtractedRelationship
   ExtractionResult


---
## 6. GraphRAGExtractor

This is the core extraction component. It sends each text chunk to the LLM
with our ontology-constrained prompt, parses the response, and stores the
extracted entities and relationships as structured objects on each node.

### Why build a custom extractor?

LlamaIndex's built-in `SchemaLLMPathExtractor` extracts entity/relationship labels
but **drops descriptions**. By building our own extractor:

- Every `EntityNode` carries a `entity_description` property
- Every `Relation` carries a `relationship_description` property
- These flow through to community summaries, making them far richer

The extractor runs **asynchronously** with `num_workers=2` — processing 2 chunks
in parallel while limiting pressure on long structured-output API requests.


In [7]:
kg_extractor = manager.extractor

print(f"✅ {type(kg_extractor).__name__} ready")
print(f"   Parallel workers: {kg_extractor.num_workers}")
print(f"   Maximum paths per chunk: {kg_extractor.max_paths_per_chunk}")

✅ GraphRAGExtractor ready
   Parallel workers: 2
   Maximum paths per chunk: 20


---
## 7. GraphRAGStore

`GraphRAGStore` extends LlamaIndex's `SimplePropertyGraphStore` with two additional
capabilities: **community detection** and **community summary generation**.

By bundling these into the store itself, the pipeline stays clean — after building
the index you just call `graph_store.build_communities()` and everything is handled.

### How community detection works here

1. Convert the property graph to a NetworkX graph
2. Run hierarchical Leiden to find entity clusters
3. For each cluster, collect all entities (+ descriptions) and relationships (+ descriptions)
4. Ask the LLM to write a briefing note for each cluster

The descriptions captured during extraction make these briefings significantly
richer than if we'd only stored bare labels.


In [8]:
# The manager creates this store when a graph is built or loaded.
print(f"✅ Graph store class ready: {GraphRAGStore.__name__}")

✅ Graph store class ready: GraphRAGStore


---
## 8. GraphRAGQueryEngine

The query engine uses a two-step approach:

1. **Per-community answering** — ask the LLM to answer the question from each
   community summary independently. If a summary isn't relevant, the LLM says so
   and we skip it. This avoids polluting the final answer with irrelevant content.

2. **Aggregation** — combine all relevant partial answers into one final,
   non-redundant response using `QUERY_LLM` (the stronger model).


In [9]:
# The query engine is instantiated after a graph has been built or loaded.
print(f"✅ Query engine class ready: {GraphRAGQueryEngine.__name__}")

✅ Query engine class ready: GraphRAGQueryEngine


---
## 9. Load Article Dataset

Load the articles previously scraped with SerpApi.

In [10]:
df = pd.read_csv(DATASET_FILE)
print(f"Number of articles: {len(df)}")
df.head()

Number of articles: 13


,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,Generative AI: Navigating intellectual property,Fully AI-generated content is ineligible for c...,Nixon Peabody,"Sep 17, 2025",https://www.nixonpeabody.com/insights/articles...,article,Generative AI is transforming creative and tec...,NaN,success
1,AI intellectual property,Artificial Intelligence and Intellectual Property,AI inventions present the current patent syste...,World Intellectual Property Organization (WIPO),NaN,https://www.wipo.int/en/web/frontier-technolog...,article,Artificial Intelligence and Intellectual Prope...,NaN,success
2,AI intellectual property,"AI, Copyright, and the Law: The Ongoing Battle...","In the past year, courts, legislators, and reg...",University of Southern California,"Feb 4, 2025",https://sites.usc.edu/iptls/2025/02/04/ai-copy...,article,By: Negar Bondari\nArtificial intelligence (AI...,NaN,success
3,AI intellectual property,AI Created It—But Do You Own It? IP Issues Exp...,AI-generated content raises complex legal ques...,DarrowEverett LLP,"Jul 7, 2025",https://darroweverett.com/ai-and-the-law-who-o...,article,Legal Insights\nAs artificial intelligence (AI...,NaN,success
4,AI intellectual property,Copyright and Artificial Intelligence | U.S. C...,Copyright and Artificial Intelligence analyzes...,Copyright Office (.gov),NaN,https://www.copyright.gov/ai/,article,Copyright and Artificial Intelligence\nSince l...,NaN,success


---
## 10. Wrap Articles as Documents

We wrap each article as a single LlamaIndex `Document` — no chunking needed since
the articles are short enough to fit in the LLM's context window.

In [11]:
nodes = manager.load_documents(DATASET_FILE, max_articles=MAX_ARTICLES)
print(f"✅ Created {len(nodes)} LlamaIndex documents, stored as nodes")

Created 3 LlamaIndex documents
✅ Created 3 LlamaIndex documents, stored as nodes


In [13]:
print(nodes[1].text)

Artificial Intelligence and Intellectual Property
Artificial intelligence (Al) is increasingly driving important developments in technology and business. It is being employed across a wide range of industries with impact on almost every aspect of the creation. The availability of large amounts of training data and advances in affordable high computing power are fueling Al's growth. Al intersects with intellectual property (IP) in a number of ways.
Eleventh session of the WIPO Conversation on AI and IP: Infrastructure for Rights Holders and Innovation
Copyright infrastructure, the generally unseen set of organizational systems, processes and technical means that support the implementation of copyright law, is essential to ensure fair protection for creators and copyright owners while allowing for technological innovation to flourish. The rise of generative AI is accelerating the need for a strong copyright infrastructure to ensure that creators are fairly protected while allowing innova

---
## 11. Build the Knowledge Graph 

Now we wire everything together and run the extraction pipeline.

`PropertyGraphIndex` handles the full workflow:
1. Passes each chunk to `GraphRAGExtractor`
2. The extractor calls the LLM with our ontology-constrained prompt
3. Parsed entities and relationships are stored in `GraphRAGStore`

-> This is the most time-consuming step!


Set `REBUILD_GRAPH = True` whenever you want to repeat the expensive extraction
workflow. When a checkpoint already exists, the default expression below loads
the graph and stored community summaries instead.

In [14]:
REBUILD_GRAPH = not CHECKPOINT_FILE.exists()

if REBUILD_GRAPH:
    # Community detection is kept for Section 12 so the original flow remains clear.
    graph_store = manager.build_knowledge_graph(
        documents=nodes,
        build_communities=False,
    )
else:
    graph_store = manager.load_knowledge_graph(CHECKPOINT_FILE)

Building knowledge graph; LLM extraction may take several minutes...


Applying transformations:   0%|          | 0/1 [00:00<?, ?it/s]

In [16]:
# Re-run extraction on a copy of one article so the stored document is not mutated.
# This cell makes an LLM call even when the graph was loaded from a checkpoint.
extraction_test = await manager.atest_extraction(document_index=2)

=== Title ===
AI, Copyright, and the Law: The Ongoing Battle Over ...

=== Raw text ===
By: Negar Bondari
Artificial intelligence (AI) is rapidly reshaping industries, but it also raises complex legal questions—particularly in the realm of copyright law. In the past year, courts, legislators, and regulators have grappled with issues such as whether AI-generated content can be copyrighted, whether AI developers are liable for using copyrighted materials to train models, and how existing intellectual property (IP) laws should adapt to new technologies. These questions have significant implications for creatives, technology companies, and legal practitioners alike.
In the rapidly evolving landscape of artificial intelligence (AI), the intersection of AI and copyright law has become a focal point of legal discourse. Recent developments have highlighted the challenges and complexities in determining the ownership and protection of AI-generated content.
1. The Current Legal Landscape for AI-

In [17]:
entities_by_type = manager.print_unique_entities()


AI_SYSTEM (8)
  AI Tools
  DABUS
  DREAMSTUDIO
  DREAMUP
  Generative AI
  Generative AI Tools
  MIDJOURNEY
  STABLE DIFFUSION

CONCEPT (21)
  AI Generated Content
  AI Generated Image
  AI IP LAW STANDARDS
  AI Models
  AI Regulation
  AI Sole Inventor
  AI Training Data
  Artificial Intelligence
  COPYRIGHTED MATERIALS
  COPYRIGHTED_WORKS
  Copyright
  Copyright Infrastructure
  Eighth Session – GenAI and IP
  Human Author
  Intellectual Property
  Ninth Session – Training the Machines – Bytes, Rights and the Copyright Conundrum
  Patent System
  Publicly Available Data
  Tenth Session – Generative AI: IP and Output
  Training Data
  WIPO Conversation on AI and IP

ENTITY (2)
  AI Generated Works
  THALER V. STABILITY AI LTD.

GOVERNMENT (6)
  BEIJING INTERNET COURT
  Beijing Internet Court
  DC Circuit
  Nigeria
  US COPYRIGHT OFFICE
  US Copyright Office

LEGAL_CASE (3)
  ANDERSEN V. STABILITY AI LTD.
  DABUS Litigation
  THALER V. PERLMUTTER

LEGISLATION (5)
  AI ACT
  EU AI Act


In [18]:
entity_details = manager.inspect_entity("DREAMUP")

Node: 'DREAMUP'  label='AI_SYSTEM'
Source: University of Southern California
Title: AI, Copyright, and the Law: The Ongoing Battle Over ...

=== Article text ===
By: Negar Bondari
Artificial intelligence (AI) is rapidly reshaping industries, but it also raises complex legal questions—particularly in the realm of copyright law. In the past year, courts, legislators, and regulators have grappled with issues such as whether AI-generated content can be copyrighted, whether AI developers are liable for using copyrighted materials to train models, and how existing intellectual property (IP) laws should adapt to new technologies. These questions have significant implications for creatives, technology companies, and legal practitioners alike.
In the rapidly evolving landscape of artificial intelligence (AI), the intersection of AI and copyright law has become a focal point of legal discourse. Recent developments have highlighted the challenges and complexities in determining the ownership and 

---
## 12. Build Communities & Generate Summaries

Now we run community detection and generate LLM summaries for each cluster.

This is the step that enables big-picture queries. Instead of searching for
similar chunks, GraphRAG queries these pre-built community briefings — each one
a synthesised overview of a topic cluster in your data.

For our AI copyright dataset, you should see communities forming around things like:
- US copyright litigation (NYT, Getty, authors suing AI companies)
- EU regulatory activity (EU AI Act, GDPR interactions)
- Training data debates (fair use arguments, dataset licensing)
- Specific AI systems and their legal exposure


In [19]:
if REBUILD_GRAPH:
    summaries = graph_store.build_communities(
        summary_llm=EXTRACTION_LLM,
        max_cluster_size=MAX_CLUSTER_SIZE,
    )
    manager.save_knowledge_graph(CHECKPOINT_FILE)
else:
    summaries = graph_store.get_community_summaries()

print(f"\n✅ {len(summaries)} community summaries ready for querying")

Running community detection...
Graph has 59 nodes, 43 edges
Found 16 communities


/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/.venv/lib/python3.12/site-packages/graspologic/partition/leiden.py:607: UserWarning: Leiden partitions do not contain all nodes from the input graph because input graph contained isolate nodes.
  warnings.warn(


  Community 0: The cluster centers on Nixon Peabody (a U.S. law firm), the World Intellectual Property Organization...
  Community 1: Briefing:
- The core entities are the US Copyright Act of 1976 (LEGISLATION) and the concept of a Hu...
  Community 2: Key players: the US Copyright Office (a U.S. government agency) and the concept of AI Generated Cont...
  Community 3: Briefing:
- The core players are the USPTO (the U.S. patent regulator) and the AI Sole Inventor conc...
  Community 4: Key entities: the U.S. Court of Appeals for the D.C. Circuit (GOVERNMENT) and the concept of AI Gene...
  Community 5: Briefing:
- The cluster centers on the Beijing Internet Court (GOVERNMENT) and the AI Generated Imag...
  Community 6: The cluster centers on three topics: the EU AI Act (EU regulation introducing transparency requireme...
  Community 7: The main elements are DABUS, an AI system, and the DABUS Litigation legal case concerned with whethe...
  Community 8: This cluster centers on Generativ

---
## 13. Visualize the Knowledge Graph

Create a d3.js network graph visualisation from the graph we built.

Open `ai_copyright_graph.html` in your browser after running this cell.


In [20]:
manager.visualize(
    graph_data_file=GRAPH_DATA_FILE,
    template_file=GRAPH_TEMPLATE_FILE,
    output_file=GRAPH_OUTPUT_FILE,
)

Graph data exported to '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/graph_data.json'
Nodes: 59 | Edges: 43 | Communities: 16
Visualization saved to '/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/ai_copyright_graph.html'


PosixPath('/home/mohamed.hanifa@dastc.stee.com/causal_graph/causalRAG/ai_copyright_graph.html')

---
## 14. Query the System

Now we run queries that would be impossible to answer reliably with standard RAG.

The `GraphRAGQueryEngine` scores each community summary for relevance, filters
irrelevant ones, and synthesizes a final answer from the best results — all using
the community briefings generated in Step 12.


In [21]:
query_engine = GraphRAGQueryEngine(
    graph_store=graph_store,
    community_llm=EXTRACTION_LLM,
    llm=QUERY_LLM,
)

print("✅ Query engine ready")

✅ Query engine ready


In [22]:
# Query 1: Big-picture thematic question
q1 = "What are the main legal arguments being made around AI copyright and training data?"
print(f"Query: {q1}")
print("=" * 70)
print(query_engine.custom_query(q1))

Query: What are the main legal arguments being made around AI copyright and training data?
The main legal arguments fall into two connected areas: **whether AI-generated outputs are protected by copyright** and **whether copyrighted works may lawfully be used to train AI systems**.

## 1. Copyrightability and authorship of AI outputs

- **Human authorship as the current gatekeeping rule:** In the United States, copyright generally requires meaningful human authorship. Under the approach reflected in *Thaler v. Perlmutter* and the U.S. Copyright Office’s guidance, material generated solely or primarily by AI is generally not copyrightable.
- **How much human input is enough:** The central dispute is whether prompting, iterative direction, selection, editing, arrangement, or curation supplies sufficient originality and control for copyright protection. If so, protection may extend only to the human-authored elements, not to purely machine-generated material.
- **Who owns protected output

In [23]:
# Query 2: Cross-entity relationship question
q2 = "Which companies are involved in AI copyright or governance disputes, and what are their positions?"
print(f"Query: {q2}")
print("=" * 70)
print(query_engine.custom_query(q2))

Query: Which companies are involved in AI copyright or governance disputes, and what are their positions?
The companies identified as involved in AI copyright or governance disputes are:

| Company | Dispute or context | Position described |
|---|---|---|
| **Getty Images** | **Getty Images v. Stability AI** | Getty Images, as a rights-holder, alleges that Stability AI infringed its copyrights by using Getty material to train AI systems. |
| **Stability AI** | Getty Images litigation; **Thaler v. Stability AI Ltd.**; **Andersen v. Stability AI Ltd.**; systems including Stable Diffusion and DreamStudio | Stability AI is identified as a defendant and as a central party in disputes over training AI on copyrighted works, licensing obligations, and liability for training data. Its specific formal defenses or policy position are not provided. |
| **The New York Times** | Lawsuit against OpenAI and Microsoft | The New York Times alleges that copyrighted Times content was used in training AI m

In [24]:
# Query 3: Comparative policy question
q3 = "How are different governments (e.g. EU, US, and UK) approaching AI governance?"
print(f"Query: {q3}")
print("=" * 70)
print(query_engine.custom_query(q3))

Query: How are different governments (e.g. EU, US, and UK) approaching AI governance?
Governments are taking different, though overlapping, approaches to AI governance. The material provided focuses mainly on intellectual property, copyright, and training data rather than broader issues such as safety, ethics, or accountability.

### European Union

The EU is pursuing a formal, **risk-based regulatory framework through the EU AI Act**. In the IP context, its approach emphasizes:

- Transparency about the data used to train general-purpose and generative AI systems.
- Compliance with copyright rules and attention to licensing or licensing-equivalent mechanisms.
- Governance obligations for developers and platforms using copyrighted works to train systems.
- Balancing AI innovation with the interests of artists, copyright holders, and other rights-holders.

The EU therefore appears to favor relatively structured, ex ante regulation, combining risk controls with transparency and copyright

In [25]:
# Query 4: Try your own question
your_question = "What is the role of fair use in AI copyright disputes?"
print(f"Query: {your_question}")
print("=" * 70)
print(query_engine.custom_query(your_question))

Query: What is the role of fair use in AI copyright disputes?
Fair use is a central legal framework in AI copyright disputes. It helps determine whether AI developers may train models on publicly available copyrighted material without obtaining licenses, or whether licensing is required. Its application depends on the specific facts of each case, so it may serve as a defense but does not automatically permit unlicensed training.

Fair use is therefore a major issue in litigation—such as *Getty Images v. Stability AI*—as well as in policy debates in the United States, the European Union, and elsewhere concerning data licensing, transparency, and governance of AI training.


In [26]:
CHECKPOINT_FILE.exists()

True

---
## 15. Summary

### What we built

| Step | Component | What it does |
|---|---|---|
| Ontology | Custom prompt | Constrains LLM to 7 entity types + 8 relationship types |
| Extraction | `GraphRAGExtractor` | Extracts entities + relationships WITH descriptions |
| Graph store | `GraphRAGStore` | Stores graph + runs Leiden community detection |
| Summaries | `_generate_summaries()` | LLM briefings per community cluster |
| Visualization | PyVis | Interactive color-coded graph by entity type |
| Querying | `GraphRAGQueryEngine` | Per-community answering + synthesis |
